In [ ]:
import matplotlib

matplotlib.use("Agg")
from pathlib import Path
import pandas as pd
import numpy as np


def project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis" / "Data").is_dir():
            return candidate
        if (candidate / "MIMIC_Delirium" / "analysis" / "Data").is_dir():
            return candidate / "MIMIC_Delirium"
    raise FileNotFoundError("Run from within the MIMIC_Delirium project directory.")


ROOT = project_root()
data_dir = ROOT / "analysis" / "Data"
df = pd.read_csv(data_dir / "mimic_delirium_analysis.csv")
cohort_flow = pd.read_csv(data_dir / "Cohort_Flow_Counts.csv")
missing = df.isna().sum()
predictors_A = ["age", "male", "sofa_24hours"]
predictors_B = predictors_A + ["mechanical_ventilation", "vasopressor_use"]
predictors_C = predictors_B + [
    "vital_sign_count_24h",
    "night_vital_ratio",
    "lab_specimen_count_24h",
    "glucose_check_count_24h",
    "input_event_count_24h",
    "procedure_count_24h",
]
feature_sets = {
    "Model A": predictors_A,
    "Model B": predictors_B,
    "Model C": predictors_C,
}
assert df["gender"].dropna().isin(["M", "F"]).all()
model_df = df.assign(male=df["gender"].map({"M": 1, "F": 0}))
model_df = model_df.dropna(subset=["delirium"] + predictors_C).reset_index(drop=True)
y = model_df["delirium"].astype(int)
assert model_df["subject_id"].notna().all()
assert model_df["subject_id"].is_unique
assert model_df["delirium"].isin([0, 1]).all()
assert np.isfinite(model_df[predictors_C].to_numpy(dtype=float)).all()
import statsmodels.api as sm
from scipy.stats import chi2
from sklearn.metrics import roc_auc_score

models = {}
comparison_rows = []
coefficient_tables = []
for name, variables in feature_sets.items():
    X = sm.add_constant(model_df[variables], has_constant="add")
    fitted = sm.Logit(y, X).fit(disp=False)
    assert fitted.mle_retvals["converged"]
    models[name] = fitted
    comparison_rows.append(
        {
            "Model": name,
            "N": int(fitted.nobs),
            "Predictors": len(variables),
            "Apparent_AUC": roc_auc_score(y, fitted.predict(X)),
            "AIC": fitted.aic,
            "BIC": fitted.bic,
        }
    )
    ci = fitted.conf_int()
    coefficient_tables.append(
        pd.DataFrame(
            {
                "Model": name,
                "Variable": fitted.params.index,
                "Coefficient": fitted.params.to_numpy(),
                "SE": fitted.bse.to_numpy(),
                "CI_lower": ci.iloc[:, 0].to_numpy(),
                "CI_upper": ci.iloc[:, 1].to_numpy(),
                "P_value": fitted.pvalues.to_numpy(),
            }
        )
    )
model_comparison = pd.DataFrame(comparison_rows)
full_coefficients = pd.concat(coefficient_tables, ignore_index=True)
lr_rows = []
for smaller, larger in [("Model A", "Model B"), ("Model B", "Model C")]:
    m0, m1 = (models[smaller], models[larger])
    lr = 2 * (m1.llf - m0.llf)
    degrees = int(m1.df_model - m0.df_model)
    lr_rows.append(
        {
            "Comparison": f"{smaller} → {larger}",
            "LR": lr,
            "df": degrees,
            "P_value": chi2.sf(lr, degrees),
        }
    )
lr_results = pd.DataFrame(lr_rows)
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
splits = list(cv.split(model_df[predictors_C], y))
fold_id = np.zeros(len(model_df), dtype=int)
held_out_count = np.zeros(len(model_df), dtype=int)
for fold, (_, test_idx) in enumerate(splits, start=1):
    fold_id[test_idx] = fold
    held_out_count[test_idx] += 1
assert (held_out_count == 1).all()
oof_predictions = {}
cv_rows = []
fold_rows = []
for name, variables in feature_sets.items():
    X = model_df[variables]
    predictions = np.full(len(model_df), np.nan)
    fold_aucs = []
    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        cv_model = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=np.inf, solver="lbfgs", max_iter=5000, tol=1e-08),
        )
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            cv_model.fit(X.iloc[train_idx], y.iloc[train_idx])
        positive_column = np.flatnonzero(cv_model.classes_ == 1)[0]
        pred = cv_model.predict_proba(X.iloc[test_idx])[:, positive_column]
        predictions[test_idx] = pred
        fold_auc = roc_auc_score(y.iloc[test_idx], pred)
        fold_aucs.append(fold_auc)
        fold_rows.append({"Model": name, "Fold": fold, "AUC": fold_auc})
    assert np.isfinite(predictions).all()
    assert ((predictions >= 0) & (predictions <= 1)).all()
    oof_predictions[name] = predictions
    cv_rows.append(
        {
            "Model": name,
            "N": len(model_df),
            "Mean_fold_AUC": np.mean(fold_aucs),
            "SD_fold_AUC": np.std(fold_aucs, ddof=1),
            "OOF_AUC": roc_auc_score(y, predictions),
        }
    )
cv_comparison = pd.DataFrame(cv_rows)
fold_results = pd.DataFrame(fold_rows)
oof_results = model_df[["subject_id", "stay_id"]].copy()
oof_results["delirium"] = y.to_numpy()
oof_results["fold"] = fold_id
for name, predictions in oof_predictions.items():
    oof_results[name.replace(" ", "_") + "_OOF"] = predictions
auc_values = cv_comparison.set_index("Model")["OOF_AUC"]
delta_auc_BC = auc_values["Model C"] - auc_values["Model B"]
y_boot = oof_results["delirium"].to_numpy()
p_B = oof_results["Model_B_OOF"].to_numpy()
p_C = oof_results["Model_C_OOF"].to_numpy()
n_boot = 2000
rng = np.random.default_rng(42)
positive_idx = np.flatnonzero(y_boot == 1)
negative_idx = np.flatnonzero(y_boot == 0)
bootstrap_deltas = np.empty(n_boot)
for b in range(n_boot):
    sampled_idx = np.concatenate(
        [
            rng.choice(positive_idx, len(positive_idx), replace=True),
            rng.choice(negative_idx, len(negative_idx), replace=True),
        ]
    )
    sampled_y = y_boot[sampled_idx]
    bootstrap_deltas[b] = roc_auc_score(sampled_y, p_C[sampled_idx]) - roc_auc_score(
        sampled_y, p_B[sampled_idx]
    )
ci_lower, ci_upper = np.percentile(bootstrap_deltas, [2.5, 97.5])
delta_auc_results = pd.DataFrame(
    [
        {
            "Comparison": "Model C minus Model B",
            "OOF_delta_AUC": delta_auc_BC,
            "CI_lower": ci_lower,
            "CI_upper": ci_upper,
            "Bootstrap_replicates": n_boot,
        }
    ]
)
from sklearn.metrics import brier_score_loss

oof_pred = oof_predictions["Model C"]
p_clip = np.clip(oof_pred, 1e-15, 1 - 1e-15)
logit_pred = np.log(p_clip / (1 - p_clip))
intercept_model = sm.GLM(
    y,
    pd.DataFrame({"const": np.ones(len(y))}),
    family=sm.families.Binomial(),
    offset=logit_pred,
).fit()
calibration_model = sm.GLM(
    y,
    pd.DataFrame({"const": np.ones(len(y)), "logit_pred": logit_pred}),
    family=sm.families.Binomial(),
).fit()
oof_brier = brier_score_loss(y, oof_pred)
calibration_intercept = intercept_model.params["const"]
calibration_slope = calibration_model.params["logit_pred"]
calibration_results = pd.DataFrame(
    [
        {
            "Model": "Model C",
            "OOF_Brier": oof_brier,
            "Calibration_intercept": calibration_intercept,
            "Calibration_slope": calibration_slope,
        }
    ]
)
adjusted_model = models["Model C"]
adjusted_ci = adjusted_model.conf_int()
regression_rows = []
for variable in predictors_C:
    X = sm.add_constant(model_df[[variable]], has_constant="add")
    crude_model = sm.Logit(y, X).fit(disp=False)
    assert crude_model.mle_retvals["converged"]
    crude_ci = crude_model.conf_int().loc[variable]
    scale = 0.1 if variable == "night_vital_ratio" else 1.0
    regression_rows.append(
        {
            "Variable": variable,
            "Unit": (
                "per 10 percentage points"
                if variable == "night_vital_ratio"
                else "per 1 unit"
            ),
            "Crude_OR": np.exp(scale * crude_model.params[variable]),
            "Crude_CI_lower": np.exp(scale * crude_ci.iloc[0]),
            "Crude_CI_upper": np.exp(scale * crude_ci.iloc[1]),
            "Crude_P": crude_model.pvalues[variable],
            "Adjusted_OR": np.exp(scale * adjusted_model.params[variable]),
            "Adjusted_CI_lower": np.exp(scale * adjusted_ci.loc[variable].iloc[0]),
            "Adjusted_CI_upper": np.exp(scale * adjusted_ci.loc[variable].iloc[1]),
            "Adjusted_P": adjusted_model.pvalues[variable],
        }
    )
regression_results = pd.DataFrame(regression_rows)
association_table = regression_results[["Variable", "Unit"]].copy()
for prefix in ["Crude", "Adjusted"]:
    association_table[f"{prefix} OR (95% CI)"] = [
        f"{odds:.3f} ({lower:.3f}–{upper:.3f})"
        for odds, lower, upper in zip(
            regression_results[f"{prefix}_OR"],
            regression_results[f"{prefix}_CI_lower"],
            regression_results[f"{prefix}_CI_upper"],
        )
    ]
    association_table[f"{prefix} P"] = regression_results[f"{prefix}_P"].map(
        lambda p: "<0.001" if p < 0.001 else f"{p:.3f}"
    )
unit_labels = {
    "age": "per year",
    "male": "male vs female",
    "sofa_24hours": "per point",
    "mechanical_ventilation": "yes vs no",
    "vasopressor_use": "yes vs no",
    "vital_sign_count_24h": "per episode",
    "night_vital_ratio": "per 10 percentage points",
    "lab_specimen_count_24h": "per specimen",
    "glucose_check_count_24h": "per check",
    "input_event_count_24h": "per record",
    "procedure_count_24h": "per record",
}
regression_results["Unit"] = regression_results["Variable"].map(unit_labels)
association_table["Unit"] = association_table["Variable"].map(unit_labels)
vasopressor_rows = []
for name in ["Model B", "Model C"]:
    fitted = models[name]
    ci = fitted.conf_int().loc["vasopressor_use"]
    vasopressor_rows.append(
        {
            "Model": name,
            "Coefficient": fitted.params["vasopressor_use"],
            "OR": np.exp(fitted.params["vasopressor_use"]),
            "CI_lower": np.exp(ci.iloc[0]),
            "CI_upper": np.exp(ci.iloc[1]),
            "P_value": fitted.pvalues["vasopressor_use"],
        }
    )
vasopressor_comparison = pd.DataFrame(vasopressor_rows)
from statsmodels.stats.outliers_influence import variance_inflation_factor

corr_matrix = model_df[predictors_C].corr(method="spearman")
X_vif = sm.add_constant(model_df[predictors_C].astype(float), has_constant="add")
vif_results = pd.DataFrame(
    [
        {"Variable": variable, "VIF": variance_inflation_factor(X_vif.to_numpy(), i)}
        for i, variable in enumerate(X_vif.columns)
        if variable != "const"
    ]
)
rho = corr_matrix.loc["input_event_count_24h", "procedure_count_24h"]
sensitivity_specs = {
    "Model C": predictors_C,
    "Without procedures": [v for v in predictors_C if v != "procedure_count_24h"],
    "Without input events": [v for v in predictors_C if v != "input_event_count_24h"],
}
sensitivity_rows = []
for name, variables in sensitivity_specs.items():
    X = sm.add_constant(model_df[variables], has_constant="add")
    fitted = models["Model C"] if name == "Model C" else sm.Logit(y, X).fit(disp=False)
    assert fitted.mle_retvals["converged"]
    sensitivity_rows.append(
        {
            "Model": name,
            "N": int(fitted.nobs),
            "Predictors": len(variables),
            "Apparent_AUC": roc_auc_score(y, fitted.predict(X)),
            "AIC": fitted.aic,
            "BIC": fitted.bic,
        }
    )
sensitivity_results = pd.DataFrame(sensitivity_rows)
from sklearn.metrics import roc_curve, confusion_matrix

fpr, tpr, thresholds = roc_curve(y, oof_pred)
best_index = np.argmax(tpr - fpr)
optimal_cutoff = thresholds[best_index]
y_pred = (oof_pred >= optimal_cutoff).astype(int)
tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
classification_metrics = pd.DataFrame(
    {
        "Metric": [
            "Threshold",
            "Sensitivity",
            "Specificity",
            "PPV",
            "NPV",
            "Accuracy",
            "F1",
        ],
        "Value": [
            optimal_cutoff,
            tp / (tp + fn),
            tn / (tn + fp),
            tp / (tp + fp),
            tn / (tn + fn),
            (tp + tn) / len(y),
            2 * tp / (2 * tp + fp + fn),
        ],
    }
)
confusion_table = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Observed negative", "Observed positive"],
    columns=["Predicted negative", "Predicted positive"],
)
from scipy.stats import mannwhitneyu, chi2_contingency

continuous_labels = {
    "age": "Age, years",
    "sofa_24hours": "SOFA score",
    "vital_sign_count_24h": "Monitoring episodes",
    "night_vital_ratio": "Night-time monitoring ratio",
    "lab_specimen_count_24h": "Laboratory specimens",
    "glucose_check_count_24h": "Glucose checks",
    "input_event_count_24h": "Input events",
    "procedure_count_24h": "Procedures",
}


def format_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


baseline_rows = []
for variable, label in continuous_labels.items():
    groups = [
        df.loc[df["delirium"] == outcome, variable].dropna() for outcome in [0, 1]
    ]
    decimals = 3 if variable == "night_vital_ratio" else 0
    values = []
    for group in groups:
        q1, median, q3 = group.quantile([0.25, 0.5, 0.75])
        values.append(f"{median:.{decimals}f} ({q1:.{decimals}f}, {q3:.{decimals}f})")
    p = mannwhitneyu(*groups, alternative="two-sided").pvalue
    baseline_rows.append(
        {
            "Characteristic": label,
            "No positive outcome": values[0],
            "Positive outcome": values[1],
            "P_value": format_p(p),
        }
    )
categorical_specs = [
    ("gender", [("F", "Female sex"), ("M", "Male sex")]),
    (
        "mechanical_ventilation",
        [(0, "Mechanical ventilation: No"), (1, "Mechanical ventilation: Yes")],
    ),
    ("vasopressor_use", [(0, "Vasopressor use: No"), (1, "Vasopressor use: Yes")]),
]
for variable, categories in categorical_specs:
    counts = pd.crosstab(df[variable], df["delirium"])
    p = chi2_contingency(counts, correction=True)[1]
    for i, (value, label) in enumerate(categories):
        values = []
        for outcome in [0, 1]:
            group = df.loc[df["delirium"] == outcome, variable].dropna()
            n = int((group == value).sum())
            values.append(f"{n} ({100 * n / len(group):.1f}%)")
        baseline_rows.append(
            {
                "Characteristic": label,
                "No positive outcome": values[0],
                "Positive outcome": values[1],
                "P_value": format_p(p) if i == 0 else "",
            }
        )
table1 = pd.DataFrame(baseline_rows)
from datetime import datetime

prefix = "mimic_delirium_analysis"
save_dir = ROOT / "analysis" / "Results"
save_dir.mkdir(parents=True, exist_ok=True)
tables = {
    "Table1_Baseline": table1,
    "Model_Comparison": model_comparison,
    "Full_Coefficients": full_coefficients,
    "Likelihood_Ratio_Tests": lr_results,
    "CrossValidation": cv_comparison,
    "Fold_AUC": fold_results,
    "OOF_Predictions": oof_results,
    "Delta_AUC_Bootstrap": delta_auc_results,
    "Bootstrap_Draws": pd.DataFrame({"Delta_AUC": bootstrap_deltas}),
    "Calibration": calibration_results,
    "Regression_Results": regression_results,
    "Association_Table": association_table,
    "Vasopressor_Comparison": vasopressor_comparison,
    "VIF": vif_results,
    "Sensitivity": sensitivity_results,
    "Classification": classification_metrics,
    "Cohort_Flow": cohort_flow,
}
for name, table in tables.items():
    table.to_csv(save_dir / f"{prefix}_{name}.csv", index=False, encoding="utf-8-sig")
corr_matrix.to_csv(save_dir / f"{prefix}_Correlation.csv", encoding="utf-8-sig")
confusion_table.to_csv(
    save_dir / f"{prefix}_Confusion_Matrix.csv", encoding="utf-8-sig"
)
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "Arial", "font.size": 11, "pdf.fonttype": 42})
fig, ax = plt.subplots(figsize=(7, 6))
colors = {"Model A": "#D55E00", "Model B": "#0072B2", "Model C": "#009E73"}
for name, predictions in oof_predictions.items():
    fpr, tpr, _ = roc_curve(y, predictions)
    auc = roc_auc_score(y, predictions)
    ax.plot(
        fpr, tpr, color=colors[name], linewidth=1.8, label=f"{name}: AUC = {auc:.4f}"
    )
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel="False-positive rate",
    ylabel="True-positive rate",
    title="Discrimination under shared 10-fold cross-validation",
)
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
for extension in ["png", "pdf"]:
    fig.savefig(
        save_dir / f"{prefix}_Figure2_ROC.{extension}", dpi=300, bbox_inches="tight"
    )
plt.close(fig)
from sklearn.calibration import calibration_curve

observed, predicted = calibration_curve(y, oof_pred, n_bins=10, strategy="quantile")
calibration_groups = pd.DataFrame(
    {"Mean_predicted_probability": predicted, "Observed_proportion": observed}
)
calibration_groups.to_csv(
    save_dir / f"{prefix}_Calibration_Groups.csv", index=False, encoding="utf-8-sig"
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1,
    label="Perfect calibration",
)
ax.plot(
    predicted,
    observed,
    "o-",
    color="#0072B2",
    linewidth=1.5,
    markersize=5,
    label="Model C",
)
ax.text(
    0.48,
    0.12,
    f"Brier score: {oof_brier:.4f}\nCalibration intercept: {calibration_intercept:.4f}\nCalibration slope: {calibration_slope:.4f}",
    transform=ax.transAxes,
    fontsize=10,
)
ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel="Mean predicted probability",
    ylabel="Observed outcome proportion",
    title="Model C calibration using OOF predictions",
)
ax.legend(loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
for extension in ["png", "pdf"]:
    fig.savefig(
        save_dir / f"{prefix}_Figure3_Calibration.{extension}",
        dpi=300,
        bbox_inches="tight",
    )
plt.close(fig)
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

plot_labels = {
    "age": "Age (per year)",
    "male": "Male vs female",
    "sofa_24hours": "SOFA score (per point)",
    "mechanical_ventilation": "Mechanical ventilation (yes vs no)",
    "vasopressor_use": "Vasopressor use (yes vs no)",
    "vital_sign_count_24h": "Monitoring episodes (per episode)",
    "night_vital_ratio": "Night-time ratio (per 10 percentage points)",
    "lab_specimen_count_24h": "Laboratory specimens (per specimen)",
    "glucose_check_count_24h": "Glucose checks (per check)",
    "input_event_count_24h": "Input events (per record)",
    "procedure_count_24h": "Procedures (per record)",
}
forest = regression_results.set_index("Variable").loc[predictors_C]
positions = np.arange(len(forest))
odds = forest["Adjusted_OR"].to_numpy()
lower = forest["Adjusted_CI_lower"].to_numpy()
upper = forest["Adjusted_CI_upper"].to_numpy()
fig, ax = plt.subplots(figsize=(12, 7))
ax.errorbar(
    odds,
    positions,
    xerr=np.vstack([odds - lower, upper - odds]),
    fmt="o",
    color="#0072B2",
    capsize=3,
    markersize=5,
)
ax.axvline(1, color="gray", linestyle="--", linewidth=1)
ax.set_yticks(positions)
ax.set_yticklabels([plot_labels[v] for v in forest.index])
ax.set_ylim(len(forest) - 0.5, -1.2)
ax.set_xscale("log")
ax.set_xlim(0.45, 4.5)
ax.xaxis.set_major_locator(FixedLocator([0.5, 0.75, 1, 1.5, 2, 3, 4]))
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
ax.xaxis.set_minor_locator(NullLocator())
for i, (value, lo, hi) in enumerate(zip(odds, lower, upper)):
    ax.text(
        1.04,
        i,
        f"{value:.3f} ({lo:.3f}–{hi:.3f})",
        transform=ax.get_yaxis_transform(),
        va="center",
        clip_on=False,
    )
ax.text(
    1.04,
    -0.8,
    "Adjusted OR (95% CI)",
    transform=ax.get_yaxis_transform(),
    va="center",
    clip_on=False,
)
ax.set_xlabel("Adjusted odds ratio (log scale)")
ax.set_title("Adjusted associations with subsequent documented delirium", pad=18)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.39, right=0.76, top=0.88, bottom=0.12)
for extension in ["png", "pdf"]:
    fig.savefig(
        save_dir / f"{prefix}_Figure4_Forest.{extension}", dpi=300, bbox_inches="tight"
    )
plt.close(fig)
from matplotlib.patches import FancyBboxPatch

flow = cohort_flow.sort_values("step").set_index("step")


def stays(step):
    return int(flow.loc[step, "n_stays"])


def patients(step):
    return int(flow.loc[step, "n_patients"])


assert patients(7) == len(df)
assert stays(7) == len(df)
assert int(flow.loc[7, "final_delirium_cases"]) == int(df["delirium"].sum())
main_labels = [
    f"All ICU stays\n{stays(1):,} stays; {patients(1):,} patients",
    f"After required joins and age ≥18 years\n{stays(3):,} stays; {patients(3):,} patients",
    f"ICU stay ≥24 hours\n{stays(4):,} stays; {patients(4):,} patients",
    f"No documented CAM-ICU positive before 24 hours\n{stays(5):,} stays; {patients(5):,} patients",
    f"At least one evaluable CAM-ICU assessment\nfrom 24 hours through ICU discharge\n{stays(6):,} stays; {patients(6):,} patients",
    f"First eligible ICU stay per patient\n{len(df):,} patients; {int(df['delirium'].sum()):,} positive outcomes",
    f"Complete-case modeling cohort\n{len(model_df):,} patients; {int(y.sum()):,} positive outcomes",
]
excluded_labels = [
    f"No stays removed\n{stays(1) - stays(3):,} excluded",
    f"ICU stay <24 hours\n{stays(3) - stays(4):,} stays excluded",
    f"Documented CAM-ICU positive before 24 hours\n{stays(4) - stays(5):,} stays excluded",
    f"No evaluable post-landmark CAM-ICU assessment\n{stays(5) - stays(6):,} stays excluded",
    f"Additional eligible stays of the same patients\n{stays(6) - stays(7):,} stays excluded",
    f"Missing model variables\n{len(df) - len(model_df):,} patients excluded",
]
fig, ax = plt.subplots(figsize=(12, 10))
ax.set(xlim=(0, 1), ylim=(0, 1))
ax.axis("off")
ys = np.linspace(0.91, 0.13, len(main_labels))
main_x, main_w, box_h = (0.03, 0.53, 0.082)
side_x, side_w = (0.63, 0.34)
center_x = main_x + main_w / 2


def draw_box(x, y, width, text):
    box = FancyBboxPatch(
        (x, y - box_h / 2),
        width,
        box_h,
        boxstyle="round,pad=0.008",
        edgecolor="#666666",
        facecolor="white",
        linewidth=1,
    )
    ax.add_patch(box)
    ax.text(x + width / 2, y, text, ha="center", va="center", fontsize=9)


for y_pos, label in zip(ys, main_labels):
    draw_box(main_x, y_pos, main_w, label)
for i, label in enumerate(excluded_labels):
    midpoint = (ys[i] + ys[i + 1]) / 2
    ax.annotate(
        "",
        xy=(center_x, ys[i + 1] + box_h / 2 + 0.008),
        xytext=(center_x, ys[i] - box_h / 2 - 0.008),
        arrowprops={"arrowstyle": "->", "color": "#666666"},
    )
    draw_box(side_x, midpoint, side_w, label)
    ax.annotate(
        "",
        xy=(side_x - 0.008, midpoint),
        xytext=(center_x, midpoint),
        arrowprops={"arrowstyle": "->", "color": "#666666"},
    )
ax.set_title("Cohort selection in MIMIC-IV", fontsize=14, pad=15)
ax.text(
    0.03,
    0.025,
    "Evaluable CAM-ICU: a recorded Positive or Negative assessment.\nCounts refer to ICU stays until selection of the first eligible stay per patient.",
    fontsize=9,
    va="bottom",
)
fig.tight_layout()
for extension in ["png", "pdf"]:
    fig.savefig(
        save_dir / f"{prefix}_Figure1_Cohort_Flow.{extension}",
        dpi=300,
        bbox_inches="tight",
    )
plt.close(fig)
import json
import platform
import hashlib
from importlib.metadata import version

expected_files = [save_dir / f"{prefix}_{name}.csv" for name in tables]
expected_files += [
    save_dir / f"{prefix}_{name}.csv"
    for name in ["Correlation", "Confusion_Matrix", "Calibration_Groups"]
]
expected_files += [
    save_dir / f"{prefix}_{figure}.{extension}"
    for figure in [
        "Figure1_Cohort_Flow",
        "Figure2_ROC",
        "Figure3_Calibration",
        "Figure4_Forest",
    ]
    for extension in ["png", "pdf"]
]
missing_files = [
    p.name for p in expected_files if not p.is_file() or p.stat().st_size == 0
]
assert not missing_files
input_hashes = {
    name: hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
    for name in ["mimic_delirium_analysis.csv", "Cohort_Flow_Counts.csv"]
}
run_info = {
    "Recorded_at": datetime.now().isoformat(),
    "Python": platform.python_version(),
    "Packages": {
        package: version(package)
        for package in [
            "numpy",
            "pandas",
            "scipy",
            "statsmodels",
            "scikit-learn",
            "matplotlib",
        ]
    },
    "Input_SHA256": input_hashes,
    "Cohort_N": len(df),
    "Model_N": len(model_df),
    "Model_events": int(y.sum()),
    "CV_folds": 10,
    "Random_seed": 42,
    "Bootstrap_replicates": n_boot,
}
(save_dir / f"{prefix}_Run_Info.json").write_text(
    json.dumps(run_info, indent=2, ensure_ascii=False), encoding="utf-8"
)


In [ ]:
"""Within-eICU model development and leave-one-hospital-out evaluation."""

from pathlib import Path
import hashlib
import platform
import warnings
from importlib.metadata import version

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import spearmanr
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

ROOT = project_root() / "analysis" / "eICU"
DATA_PATH = ROOT / "data" / "eICU_model_ABC_complete_case_v1.0.csv"
RESULTS_DIR = ROOT / "results"
MODEL_A = ["age", "male", "aps"]
MODEL_B = MODEL_A + ["day1_intubation", "vasopressor_24h"]
CARE_PROCESS = [
    "monitoring_episodes_24h",
    "nighttime_monitoring_ratio",
    "laboratory_testing_episodes_24h",
    "glucose_checks_24h",
    "input_events_24h",
    "procedure_events_24h",
]
MODELS = {"Model A": MODEL_A, "Model B": MODEL_B, "Model C": MODEL_B + CARE_PROCESS}
SEED = 20260907
N_BOOT = 1000


def build_model():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=np.inf, solver="lbfgs", max_iter=5000, tol=1e-8),
    )


def fit_predict(train, test, variables):
    model = build_model()
    model.fit(train[variables], train["outcome"])
    return model, model.predict_proba(test[variables])[:, 1]


def calibration(y, p):
    logit = np.log(np.clip(p, 1e-15, 1 - 1e-15) / (1 - np.clip(p, 1e-15, 1 - 1e-15)))
    intercept = sm.GLM(
        y, np.ones((len(y), 1)), offset=logit, family=sm.families.Binomial()
    ).fit()
    slope = sm.GLM(y, sm.add_constant(logit), family=sm.families.Binomial()).fit()
    if not intercept.converged or not slope.converged:
        raise RuntimeError("Calibration model did not converge.")
    return float(intercept.params[0]), float(slope.params[1])


def metrics(y, p):
    return {"AUC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p)}


def summarize(frame, columns):
    return pd.DataFrame(
        [
            {
                "Metric": c,
                "N_Hospitals": len(frame),
                "Median": frame[c].median(),
                "P25": frame[c].quantile(0.25),
                "P75": frame[c].quantile(0.75),
                "Min": frame[c].min(),
                "Max": frame[c].max(),
            }
            for c in columns
        ]
    )


def save(frame, name):
    frame.to_csv(RESULTS_DIR / f"eICU_{name}_v1.0.csv", index=False)


def evaluate_hospitals(df, models, threshold):
    counts = df.groupby("hospitalid").outcome.agg(["size", "sum"])
    eligible = counts.index[
        (counts["sum"] >= threshold) & (counts["size"] - counts["sum"] >= threshold)
    ]
    records = []
    for hospital in eligible:
        train = df.loc[df.hospitalid != hospital]
        test = df.loc[df.hospitalid == hospital]
        y = test.outcome.to_numpy()
        row = {
            "hospitalid": hospital,
            "n": len(test),
            "events": int(y.sum()),
            "non_events": int(len(y) - y.sum()),
            "event_rate": y.mean(),
        }
        for name, variables in models.items():
            _, p = fit_predict(train, test, variables)
            row.update({f"{name}_{k}": v for k, v in metrics(y, p).items()})
            intercept, slope = calibration(y, p)
            row[f"{name}_Calibration_Intercept"] = intercept
            row[f"{name}_Calibration_Slope"] = slope
            row[f"{name}_Mean_Predicted_Risk"] = p.mean()
        row["Delta_AUC_A_B"] = row["Model B_AUC"] - row["Model A_AUC"]
        row["Delta_AUC_B_C"] = row["Model C_AUC"] - row["Model B_AUC"]
        row["Delta_Brier_C_minus_B"] = row["Model C_Brier"] - row["Model B_Brier"]
        records.append(row)
    return pd.DataFrame(records)


def main():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    warnings.simplefilter("error", ConvergenceWarning)
    df = pd.read_csv(DATA_PATH)
    required = ["outcome", "hospitalid", "day1_any_ventilation"] + MODELS["Model C"]
    if (
        df[required].isna().any().any()
        or not np.isfinite(df[required].to_numpy()).all()
    ):
        raise ValueError("Missing or nonfinite analysis values.")
    for column in ["patientunitstayid", "uniquepid"]:
        if df[column].isna().any() or not df[column].is_unique:
            raise ValueError(f"Nonunique or missing {column}.")
    for column in [
        "outcome",
        "male",
        "day1_intubation",
        "day1_any_ventilation",
        "vasopressor_24h",
    ]:
        if not df[column].isin([0, 1]).all():
            raise ValueError(f"Invalid binary values in {column}.")
    if len(df) != 10015 or df.outcome.sum() != 1688 or df.hospitalid.nunique() != 49:
        raise ValueError("Unexpected cohort counts.")
    y = df.outcome.to_numpy()
    apparent, fitted = {}, {}
    for name, variables in MODELS.items():
        fitted[name], p = fit_predict(df, df, variables)
        apparent[name] = metrics(y, p)
    rng = np.random.default_rng(SEED)
    draws = []
    for b in range(N_BOOT):
        sample = df.iloc[rng.integers(0, len(df), len(df))]
        row = {"iteration": b + 1}
        for name, variables in MODELS.items():
            model, p_test = fit_predict(sample, df, variables)
            p_boot = model.predict_proba(sample[variables])[:, 1]
            boot, test = metrics(sample.outcome, p_boot), metrics(y, p_test)
            for metric in ["AUC", "Brier"]:
                key = metric.lower()
                row[f"{name}_{key}_boot"] = boot[metric]
                row[f"{name}_{key}_test"] = test[metric]
                sign = 1 if metric == "AUC" else -1
                row[f"{name}_{key}_optimism"] = sign * (boot[metric] - test[metric])
        for a, bname, tag in [
            ("Model A", "Model B", "A_B"),
            ("Model B", "Model C", "B_C"),
        ]:
            for kind in ["boot", "test"]:
                row[f"delta_auc_{tag}_{kind}"] = (
                    row[f"{bname}_auc_{kind}"] - row[f"{a}_auc_{kind}"]
                )
        draws.append(row)
    boot = pd.DataFrame(draws)
    save(boot, "bootstrap_draws")
    corrected = []
    for name, variables in MODELS.items():
        row = {
            "Model": name,
            "N": len(df),
            "Events": int(y.sum()),
            "Hospitals": df.hospitalid.nunique(),
        }
        for metric in ["AUC", "Brier"]:
            optimism = boot[f"{name}_{metric.lower()}_optimism"].mean()
            row[f"Apparent_{metric}"] = apparent[name][metric]
            row[f"Mean_{metric}_Optimism"] = optimism
            row[f"Corrected_{metric}"] = (
                apparent[name][metric] + (-1 if metric == "AUC" else 1) * optimism
            )
        scaler, model = fitted[name].steps[0][1], fitted[name].steps[1][1]
        raw_coef = model.coef_[0] / scaler.scale_
        row["Coefficient_const"] = model.intercept_[0] - np.dot(raw_coef, scaler.mean_)
        row.update({f"Coefficient_{v}": c for v, c in zip(variables, raw_coef)})
        row.update(
            {
                "Bootstrap_replicates": N_BOOT,
                "Seed": SEED,
                "Penalty": "none",
                "Input_SHA256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
                "Python": platform.python_version(),
                "Software": ", ".join(
                    f"{m} {version(m)}"
                    for m in ["numpy", "pandas", "scipy", "scikit-learn", "statsmodels"]
                ),
            }
        )
        corrected.append(row)
    save(pd.DataFrame(corrected), "bootstrap_corrected_performance")
    deltas = []
    for a, bname, tag in [("Model A", "Model B", "A_B"), ("Model B", "Model C", "B_C")]:
        delta = apparent[bname]["AUC"] - apparent[a]["AUC"]
        optimism = (
            boot[f"delta_auc_{tag}_boot"] - boot[f"delta_auc_{tag}_test"]
        ).mean()
        samples = boot[f"delta_auc_{tag}_boot"]
        deltas.append(
            {
                "Comparison": f"{a} to {bname}",
                "Apparent_Delta_AUC": delta,
                "Mean_Optimism": optimism,
                "Corrected_Delta_AUC": delta - optimism,
                "Bootstrap_Median_Delta_AUC": samples.median(),
                "Bootstrap_P2.5": samples.quantile(0.025),
                "Bootstrap_P97.5": samples.quantile(0.975),
                "Percentile_interpretation": "Distribution of resampled apparent increments, not a CI for the corrected increment",
            }
        )
    save(pd.DataFrame(deltas), "delta_AUC_bootstrap")
    all_hospitals = evaluate_hospitals(df, MODELS, 10)
    primary = all_hospitals.loc[
        (all_hospitals.events >= 20) & (all_hospitals.non_events >= 20)
    ].copy()
    auc_columns = [f"Model {m}_AUC" for m in "ABC"] + ["Delta_AUC_A_B", "Delta_AUC_B_C"]
    save(primary, "IECV_hospital_results")
    save(summarize(primary, auc_columns), "IECV_summary")
    save(all_hospitals, "IECV_threshold10_hospital_results")
    save(summarize(all_hospitals, auc_columns), "IECV_threshold10_summary")
    comparison = []
    for threshold, frame in [(20, primary), (10, all_hospitals)]:
        comparison.append(
            {
                "Minimum_events_and_non_events": threshold,
                "Hospitals": len(frame),
                "Patients": int(frame.n.sum()),
                "Events": int(frame.events.sum()),
                "Median_Delta_AUC_B_C": frame.Delta_AUC_B_C.median(),
                "P25": frame.Delta_AUC_B_C.quantile(0.25),
                "P75": frame.Delta_AUC_B_C.quantile(0.75),
                "Positive_hospitals": int((frame.Delta_AUC_B_C > 0).sum()),
            }
        )
    save(pd.DataFrame(comparison), "IECV_threshold20_vs_10")
    save(primary, "IECV_calibration_hospital_results")
    cal_columns = [
        f"Model {m}_{metric}"
        for metric in [
            "AUC",
            "Brier",
            "Calibration_Intercept",
            "Calibration_Slope",
            "Mean_Predicted_Risk",
        ]
        for m in "BC"
    ] + ["Delta_AUC_B_C", "Delta_Brier_C_minus_B"]
    save(summarize(primary, cal_columns), "IECV_calibration_summary")
    save(
        summarize(
            primary,
            [
                "event_rate",
                "Model B_Mean_Predicted_Risk",
                "Model C_Mean_Predicted_Risk",
            ],
        ),
        "IECV_observed_vs_predicted_risk",
    )
    correlations = []
    for col in [
        "Model B_AUC",
        "Model C_AUC",
        "Delta_AUC_B_C",
        "Delta_Brier_C_minus_B",
        "Model B_Calibration_Slope",
        "Model C_Calibration_Slope",
    ]:
        r, p = spearmanr(primary.event_rate, primary[col])
        correlations.append(
            {"Metric": col, "Spearman_rho_with_event_rate": r, "P_value": p}
        )
    save(pd.DataFrame(correlations), "IECV_heterogeneity_correlations")
    sensitivity = {
        k: [("day1_any_ventilation" if v == "day1_intubation" else v) for v in values]
        for k, values in MODELS.items()
    }
    sens = evaluate_hospitals(df, sensitivity, 20)
    save(sens, "sensitivity_anyVent_IECV_hospitals")
    save(summarize(sens, auc_columns), "sensitivity_anyVent_IECV_summary")
    pooled = []
    for name, variables in sensitivity.items():
        _, p = fit_predict(df, df, variables)
        pooled.append({"Model": name, **metrics(y, p)})
    save(pd.DataFrame(pooled), "sensitivity_anyVent_pooled")
    save(
        pd.DataFrame(
            [
                {
                    "Specification": label,
                    "Hospitals": len(frame),
                    "Median_Delta_AUC_B_C": frame.Delta_AUC_B_C.median(),
                    "P25": frame.Delta_AUC_B_C.quantile(0.25),
                    "P75": frame.Delta_AUC_B_C.quantile(0.75),
                    "Positive_hospitals": int((frame.Delta_AUC_B_C > 0).sum()),
                }
                for label, frame in [
                    ("Day-one intubation", primary),
                    ("Any day-one ventilation", sens),
                ]
            ]
        ),
        "primary_vs_anyVent_sensitivity",
    )
    print(
        f"Completed {N_BOOT} bootstrap replicates and hospital validation in {len(primary)} primary hospitals."
    )


if __name__ == "__main__":
    with threadpool_limits(limits=1):
        main()
